# Génération de masques de sol
<br/>
Dans ce notebook, nous allons générer un masque de sol sur Tournefeuille.<br/>
Pour cela, nous allons calculer le NDVI, le NDWI, les seuiller, puis les merger.<br/>
<br/>
Import et installation des librairies nécessaires

In [ ]:
try:
    import openeo
except:
    !pip install openeo
    import openeo

In [ ]:
# Connexion à l'aggrégateur
con = openeo.connect("https://aggregator.geodes-lab.fr")
con.authenticate_oidc() 

## Création du datacube et du process graph

**Etape 1 :** Utiliser le process "load_collection" pour charger les données.

In [ ]:
interval = ["2022-03-01","2022-04-01"]
bbox = {"west": 1.2905, "east": 1.3771, "south": 43.5544, "north": 43.5950, "crs": 4326}

df = con.load_collection(
    "S2-L2A-COG", 
    temporal_extent=interval,
    spatial_extent=bbox,
    bands=["B3", "B4", "B8"],
    max_cloud_cover=40
)
df

**Etape 2 :** Création du reducer

In [ ]:
from openeo.processes import if_

def reducer(data):
    # On extrait les bandes
    green = data.array_element(index=0)
    red   = data.array_element(index=1)
    nir   = data.array_element(index=2)

    # On calcule le NDVI et le NDWI
    ndvi = (nir - red) / (nir + red)
    ndwi = (green - nir) / (green + nir)

    # On applique les seuils
    # 2=végétation haute, 1=eau, 0=autre
    res = if_(
        ndwi > 0,
        1,
        if_(ndvi > 0.6, 2, 0)
    )

    return res

df_reduce = (
    df.reduce_dimension(dimension="bands", reducer=reducer)
)
df_reduce

**Etape 3 :** On ajoute le save_result avec le profil voulu.

In [ ]:
df_result = df_reduce.save_result(
    format="COG",
    options={"dtype": "uint8", "nodata": 255}
)
df_result

## Création et lancement du job

In [ ]:
job_df = df_result.create_job(title="Masques de sol")

In [ ]:
job_df.start_and_wait()  # environ 1 min

**Une fois que le job est terminé (status=finished), on récupère les résultats**

In [ ]:
# Aperçu des résultats
results = job_df.get_results()
results

In [ ]:
# Téléchargement
results.download_files("./result_mask/")

## Pour la visualisation, on récupère aussi l'image de base

In [ ]:
df_collection = con.load_collection(
    "S2-L2A-COG", 
    temporal_extent=interval,
    spatial_extent=bbox,
    bands=["B4", "B3", "B2"],
    max_cloud_cover=40
)
job_df_collection = df_collection.create_job(title="Tournefeuille", out_format="COG")
job_df_collection.start_and_wait()

In [ ]:
results_collection = job_df_collection.get_results()
results_collection.download_files("./result_raw/")

## Visualisation des résultats

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

In [ ]:
file_path = "openeo_2022-03-26T000000_cog.tif"

with rasterio.open(f"result_mask/{file_path}") as src:
    mask = src.read(1)

with rasterio.open(f"result_raw/{file_path}") as src:
    raw = src.read()
    
# Normalisation
rgb = np.moveaxis(raw, 0, -1)
low, high = np.percentile(rgb, [2, 98])
rgb = np.clip((rgb - low) / (high - low), 0, 1)

# Figures
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cmap = ListedColormap(["white", "dodgerblue", "green"])
axes[0].imshow(mask, vmin=0, vmax=2, cmap=cmap)
axes[0].set_title("Masque")

axes[1].imshow(rgb, vmax=13000)
axes[1].set_title("NIR")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Clean

import shutil

shutil.rmtree("./result_mask/")
shutil.rmtree("./result_raw/")